## Generate BlueSTARR training data

Whole-genome STARR-seq input (DNA, 3 reps) and output (RNA, 3 reps) data are summarized in 300bp overlapping windows (50bp step). We use the bwtool command-line tool to extract counts per window and awk to compute the average coverage per window.

In [ ]:
%%bash
bwtool window 300 \
    <SAMPLE>.<REP>.bw \
    -step=50 \
    -skip-NA \
| awk -vOFS="\t" 'NR==1{ split($4, vv,","); nwins=length(vv)}{tot=0; split($4, vv,","); for (ii=1; ii<=nwins; ii+=1){tot+=vv[ii]} print $1,$2,$3,tot/nwins}' \
| awk '$4!=0' \
> <SAMPLE>.<REP>.w300s50.bdg

Next, we find the set of windows with data for all replicates/conditions:

In [ ]:
import numpy as np
import pandas as pd
bedgraphs = [
<SAMPLE>.<REP1>.w300s50.bdg,
<SAMPLE>.<REP2>.w300s50.bdg,
<SAMPLE>.<REP3>.w300s50.bdg
]
df = pd.read_csv(bedgraphs[0], sep='\t', names = ['chrom','start','end','count'])
df.index = df['chrom'] + "_" + df['start'].astype('str') + "_" + df['end'].astype('str')
for ii in bedgraphs[1:]:
    df_tmp = pd.read_csv(ii, sep='\t', names = ['chrom','start','end','count'])
    df_tmp.index = df['chrom'] + "_" + df['start'].astype('str') + "_" + df['end'].astype('str')
    df = df.index.join(df_tmp.index, how='inner')

Use the common windows to filter the windows:

In [ ]:
for ii in bedgraphs:
    df_tmp = pd.read_csv(ii, sep='\t', names = ['chrom','start','end','count'])
    df_tmp.index = df['chrom'] + "_" + df['start'].astype('str') + "_" + df['end'].astype('str')
    df_tmp.join(pd.DataFrame(index=df), how='inner').to_csv(ii.replace('.w300s50.bdg','.w300s50.in_common_win.bdg'), sep='\t', index = False)

Next, merge across all samples and replicates, selecting only windows with enough average counts (100) across both input and output:


In [ ]:
%%bash
THRES=150;
paste -d"\t" \
    <SAMPLE_DNA>.*.w300s50.in_common_win.bdg \
    <SAMPLE_RNA>.*.w300s50.in_common_win.bdg \
|awk -vTHRES=${THRES} -vOFS="\t" 'BEGIN{print"chrom\tstart\tend\tinput_rep1\tinput_rep2\tinput_rep3\toutput_rep1\toutput_rep2\toutput($4+$8+$12>THRES) && ($16+$20+$24>THRES){print $1,$2,$3,$4,$8,$12,$16,$20,$24}' \
> combined.input_and_output.gt_${THRES}.bdg

Compute a simple RNA/DNA log2fc with pseudocounts, in python:

In [ ]:
import numpy as np
import pandas as pd
thres = 150
df = pd.read_csv(f'combined.input_and_output.gt_{thres}.bdg', sep='\t')
df = df.astype(int, errors='ignore')
df.iloc[:, 3:] = df.iloc[:, 3:]/df.iloc[:, 3:].sum()*1e6

log2FC_tmp = (np.log2((0.001+df.iloc[:, 6:].mean(axis=1)) / (0.001+df.iloc[:, 3:6].mean(axis=1))))

df = None
df = pd.read_csv(f'combined.input_and_output.gt_{thres}.bdg', sep='\t')
df = df.astype(int, errors='ignore')
df['log2FC'] = log2FC_tmp
df.to_csv(f'combined.input_and_output.gt_{thres}.log2FC.txt', sep='\t', index=False, float_format='%.5f')

To add the nucleotide sequence information, we extract the windows sequences from the reference genome:

In [ ]:
%%bash
tail -n+2 combined.input_and_output.gt_${THRES}.log2FC.txt \
| cut -f1-3 \
| bedtools getfasta \
    -fi GCA_000001405.15_GRCh38_full_analysis_set.fna \
    -bed stdin \
> combined.input_and_output.gt_${THRES}.sequences.fa

And finally, add it to the counts and log2fc table file:

In [ ]:
%%bash
THRES=150
paste -d"\t" combined.input_and_output.gt_${THRES}.log2FC.txt \
    <(cat <(echo sequence) <(cat
combined.input_and_output.gt_${THRES}.sequences.fa | awk 'NR%2==0')) \
    | gzip -c \
    > combined.input_and_output.gt_${THRES}.log2FC.sequence.txt.gz